# 05 · Lakebase operational serving (Delta ⇄ Postgres)

The RG case queue is **operational**: analysts claim cases, log interventions and close them, and SLA timers have to be exact.
That's OLTP work, so it lives in **Lakebase Postgres**. Analytics stays in Delta.

This notebook runs daily after scoring. It does two things:
1. **Lakebase → Delta:** copies the analysts' interventions into `rg_intervention_audit`. This gives the regulator an audit trail, and the outcomes can serve as future training labels.
2. **Delta → Lakebase:** upserts HIGH/MEDIUM risk scores and GenAI briefs into `rg_ops.risk_cases`. Analyst-owned fields (status, assignee) are never overwritten.

In [ ]:
%pip install "psycopg[binary]>=3.2" -q
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
dbutils.widgets.text("catalog", "<catalog>")
dbutils.widgets.text("schema", "sportsbook_rg")
dbutils.widgets.text("lakebase_endpoint", "projects/rg-early-warning/branches/production/endpoints/primary")
dbutils.widgets.text("app_sp_role", "")  # app service principal client id, granted DML on rg_ops
CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
EP, APP_ROLE = dbutils.widgets.get("lakebase_endpoint"), dbutils.widgets.get("app_sp_role").strip()
spark.sql(f"USE {CATALOG}.{SCHEMA}")

{'text/plain': 'DataFrame[]'}

In [ ]:
import psycopg, json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.api_client.do("GET", f"/api/2.0/postgres/{EP}")["status"]["hosts"]["host"]
token = w.api_client.do("POST", "/api/2.0/postgres/credentials", body={"endpoint": EP})["token"]
user = w.current_user.me().user_name
conn = psycopg.connect(host=host, dbname="databricks_postgres", user=user, password=token, sslmode="require", autocommit=True)
print("Connected to Lakebase:", conn.execute("select version()").fetchone()[0][:60])

Connected to Lakebase: PostgreSQL 17.11 (0bfbbeb) on x86_64-pc-linux-gnu, compiled 


In [ ]:
DDL = """
CREATE SCHEMA IF NOT EXISTS rg_ops;
CREATE TABLE IF NOT EXISTS rg_ops.risk_cases (
  player_id          text PRIMARY KEY,
  market             text NOT NULL,
  risk_score         double precision NOT NULL,
  risk_tier          text NOT NULL,
  reason_codes       text[],
  summary            text,
  recommended_action text,
  urgency            text,
  model_version      int,
  status             text NOT NULL DEFAULT 'OPEN' CHECK (status IN ('OPEN','IN_REVIEW','ACTIONED','CLOSED_NO_ACTION')),
  assigned_to        text,
  opened_at          timestamptz NOT NULL DEFAULT now(),
  sla_due_at         timestamptz NOT NULL,
  scored_at          timestamptz NOT NULL,
  updated_at         timestamptz NOT NULL DEFAULT now()
);
CREATE INDEX IF NOT EXISTS risk_cases_queue_idx ON rg_ops.risk_cases (market, status, sla_due_at);
CREATE TABLE IF NOT EXISTS rg_ops.interventions (
  intervention_id bigserial PRIMARY KEY,
  player_id       text NOT NULL REFERENCES rg_ops.risk_cases(player_id),
  action          text NOT NULL,
  note            text,
  analyst         text NOT NULL,
  new_status      text NOT NULL,
  created_at      timestamptz NOT NULL DEFAULT now()
);
"""
conn.execute(DDL)
if APP_ROLE:
    conn.execute(f'GRANT USAGE ON SCHEMA rg_ops TO "{APP_ROLE}"')
    conn.execute(f'GRANT SELECT, INSERT, UPDATE ON ALL TABLES IN SCHEMA rg_ops TO "{APP_ROLE}"')
    conn.execute(f'GRANT USAGE ON ALL SEQUENCES IN SCHEMA rg_ops TO "{APP_ROLE}"')
    print("Granted rg_ops DML to app role", APP_ROLE)

Granted rg_ops DML to app role <app_sp_client_id>


 ### 1 · Lakebase → Delta: intervention audit trail

In [ ]:
import pandas as pd
rows = conn.execute("""SELECT i.intervention_id, i.player_id, c.market, i.action, i.note, i.analyst, i.new_status, i.created_at
                       FROM rg_ops.interventions i JOIN rg_ops.risk_cases c USING (player_id)""").fetchall()
cols = ["intervention_id", "player_id", "market", "action", "note", "analyst", "new_status", "created_at"]
spark.sql("""CREATE TABLE IF NOT EXISTS rg_intervention_audit (intervention_id BIGINT, player_id STRING, market STRING, action STRING,
             note STRING, analyst STRING, new_status STRING, created_at TIMESTAMP)
             COMMENT 'Analyst interventions copied from Lakebase rg_ops.interventions: regulatory audit trail and feedback labels.'""")
if rows:
    spark.createDataFrame(pd.DataFrame(rows, columns=cols)).createOrReplaceTempView("new_interventions")
    spark.sql("""MERGE INTO rg_intervention_audit t USING new_interventions s ON t.intervention_id = s.intervention_id
                 WHEN NOT MATCHED THEN INSERT *""")
print("Interventions in Lakebase:", len(rows), "| in Delta audit:", spark.table("rg_intervention_audit").count())

Interventions in Lakebase: 1 | in Delta audit: 1


 ### 2 · Delta → Lakebase: publish today's cases

In [ ]:
cases = spark.sql("""
  SELECT s.player_id, s.market, s.risk_score, s.risk_tier, s.reason_codes, b.summary, b.recommended_action,
         coalesce(b.urgency, CASE WHEN s.risk_tier = 'HIGH' THEN 'WITHIN_72H' ELSE 'THIS_WEEK' END) AS urgency,
         s.model_version, s.scored_at
  FROM player_risk_scores s LEFT JOIN rg_case_briefs b USING (player_id)
  WHERE s.risk_tier IN ('HIGH', 'MEDIUM')""").toPandas()

UPSERT = """
INSERT INTO rg_ops.risk_cases AS c (player_id, market, risk_score, risk_tier, reason_codes, summary, recommended_action, urgency,
                                    model_version, scored_at, sla_due_at)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,
        now() + CASE %s WHEN 'WITHIN_24H' THEN interval '24 hours' WHEN 'WITHIN_72H' THEN interval '72 hours' ELSE interval '7 days' END)
ON CONFLICT (player_id) DO UPDATE SET
  risk_score = EXCLUDED.risk_score, risk_tier = EXCLUDED.risk_tier, reason_codes = EXCLUDED.reason_codes,
  summary = coalesce(EXCLUDED.summary, c.summary), recommended_action = coalesce(EXCLUDED.recommended_action, c.recommended_action),
  urgency = EXCLUDED.urgency, model_version = EXCLUDED.model_version, scored_at = EXCLUDED.scored_at, updated_at = now(),
  -- a more urgent assessment tightens the SLA; it never extends it
  sla_due_at = LEAST(c.sla_due_at, c.opened_at + (EXCLUDED.sla_due_at - now()))
"""
with conn.cursor() as cur:
    cur.executemany(UPSERT, [
        (r.player_id, r.market, float(r.risk_score), r.risk_tier, list(r.reason_codes) if r.reason_codes is not None else [],
         r.summary, r.recommended_action, r.urgency, int(r.model_version), r.scored_at.to_pydatetime(), r.urgency)
        for r in cases.itertuples()])
print(f"Upserted {len(cases):,} cases into Lakebase rg_ops.risk_cases")

# Queue rule: an OPEN case whose player is no longer HIGH/MEDIUM after re-scoring is auto-resolved (attributed to the system)
stale = conn.execute("""UPDATE rg_ops.risk_cases SET status = 'CLOSED_NO_ACTION', risk_tier = 'LOW', assigned_to = 'system:rescore',
                        updated_at = now() WHERE status = 'OPEN' AND NOT (player_id = ANY(%s)) RETURNING player_id""",
                     (list(cases.player_id),)).fetchall()
print(f"Auto-resolved {len(stale):,} open cases that fell below the MEDIUM threshold")

Upserted 1,159 cases into Lakebase rg_ops.risk_cases
Auto-resolved 0 open cases that fell below the MEDIUM threshold


In [ ]:
display(pd.DataFrame(conn.execute("""
  SELECT market, risk_tier, status, count(*) AS cases, min(sla_due_at) AS earliest_sla
  FROM rg_ops.risk_cases GROUP BY 1,2,3 ORDER BY 1,2,3""").fetchall(),
  columns=["market", "risk_tier", "status", "cases", "earliest_sla"]))

| market | risk_tier | status | cases | earliest_sla |
|---|---|---|---|---|
| BR | HIGH | OPEN | 102 | 2026-11-02T22:53:07.042Z |
| BR | LOW | CLOSED_NO_ACTION | 195 | 2026-11-04T22:53:07.042Z |
| BR | MEDIUM | OPEN | 227 | 2026-11-04T22:53:07.042Z |
| CY | HIGH | OPEN | 34 | 2026-11-02T23:14:38.901Z |
| CY | LOW | CLOSED_NO_ACTION | 71 | 2026-11-04T22:53:07.042Z |
| CY | MEDIUM | OPEN | 76 | 2026-11-04T22:53:07.042Z |
| GR | HIGH | ACTIONED | 1 | 2026-11-02T22:53:07.042Z |
| GR | HIGH | OPEN | 95 | 2026-11-02T22:53:07.042Z |
| GR | LOW | CLOSED_NO_ACTION | 189 | 2026-11-04T22:53:07.042Z |
| GR | MEDIUM | OPEN | 230 | 2026-11-04T22:53:07.042Z |
| MX | HIGH | OPEN | 58 | 2026-11-02T23:14:38.901Z |
| MX | LOW | CLOSED_NO_ACTION | 90 | 2026-11-04T22:53:07.042Z |
| MX | MEDIUM | OPEN | 157 | 2026-11-04T22:53:07.042Z |
| RO | HIGH | OPEN | 58 | 2026-11-02T22:53:07.042Z |
| RO | LOW | CLOSED_NO_ACTION | 155 | 2026-11-04T22:53:07.042Z |
| RO | MEDIUM | OPEN | 121 | 2026-11-04T22:53:07.042Z |

In [ ]:
conn.close()